In [1]:
pip install bs4

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.iplwinners.net/auction/"
res = requests.get(url)

soup = BeautifulSoup(res.text, "html.parser")

# Find table
table = soup.find("table")

rows = table.find_all("tr")

data = []
headers = []

# Extract headers
for th in rows[0].find_all("th"):
    headers.append(th.text.strip())

# Extract rows
for row in rows[1:]:
    cols = [td.text.strip() for td in row.find_all("td")]
    if cols:
        data.append(cols)

df = pd.DataFrame(data, columns=headers)

print(df.head())

df.to_csv("ipl_auction_team_summary.csv", index=False)

  Season       Top Buy 1     Price  Team      Top Buy 2     Price  Team
0   2025    Rishabh Pant  27.00 Cr   LSG   Shreyas Iyer  26.75 Cr  PBKS
1   2024  Mitchell Starc  24.75 Cr   KKR    Pat Cummins  20.50 Cr   SRH
2   2023      Sam Curran   18.5 Cr  PBKS  Cameron Green   17.5 Cr    MI
3   2022    Ishan Kishan  15.25 Cr    MI  Deepak Chahar     14 Cr   CSK
4   2021    Chris Morris  16.25 Cr    RR  Kyle Jamieson     15 Cr   RCB


In [16]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://en.wikipedia.org/wiki/List_of_2025_Indian_Premier_League_personnel_changes"
HEADERS = {"User-Agent": "Mozilla/5.0"}

resp = requests.get(URL, headers=HEADERS)
soup = BeautifulSoup(resp.text, "html.parser")

all_rows = []

tables = soup.find_all("table", class_="wikitable")

for table in tables:
    caption = table.find("caption")
    if not caption:
        continue

    team_name = caption.get_text(strip=True)

    header_row = table.find("tr")
    col_headers = [th.get_text(strip=True) for th in header_row.find_all("th")]

    if "Player" not in col_headers:
        continue

    for row in table.find_all("tr")[1:]:
        tds = row.find_all("td")
        if not tds:
            continue

        row_data = [td.get_text(strip=True) for td in tds]

        row_dict = {"Team": team_name}
        for i, col in enumerate(col_headers):
            row_dict[col] = row_data[i] if i < len(row_data) else None

        all_rows.append(row_dict)

df = pd.DataFrame(all_rows)

# Debug: print columns and first few rows to see what we got
print("Columns found:", df.columns.tolist())
print(df.head())

# Convert to string first, then extract
if "Salary" in df.columns:
    df["Salary"] = df["Salary"].astype(str).str.extract(r"(₹[\d.]+ crore)")
else:
    # Salary column might have a different name — print what we have
    print("\nNo 'Salary' column found. Available columns:", df.columns.tolist())

print(df.to_string(index=False))
df.to_csv("ipl_2025_retained.csv", index=False)
print("\nSaved to ipl_2025_retained.csv")

Columns found: ['Team', 'No.', 'Player', 'Nationality', 'Salary', 'Auctioned/retention price', 'Reason', 'Withdrawal announcement date', 'Replacement player', "Replacement player's price[74]", 'Signing date', 'Ref.']
                      Team No.     Player               Nationality  Salary  \
0  Chennai Super Kings[20]   1      India  ₹18crore(US$2.1 million)     NaN   
1  Chennai Super Kings[20]   2      India  ₹18crore(US$2.1 million)     NaN   
2  Chennai Super Kings[20]   3  Sri Lanka  ₹13crore(US$1.5 million)     NaN   
3  Chennai Super Kings[20]   4      India  ₹12crore(US$1.4 million)     NaN   
4  Chennai Super Kings[20]   5      India       ₹4crore(US$470,000)     NaN   

  Auctioned/retention price Reason Withdrawal announcement date  \
0                       NaN    NaN                          NaN   
1                       NaN    NaN                          NaN   
2                       NaN    NaN                          NaN   
3                       NaN    NaN      

In [21]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

URL = "https://en.wikipedia.org/wiki/List_of_2025_Indian_Premier_League_personnel_changes"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

resp = requests.get(URL, headers=HEADERS)
soup = BeautifulSoup(resp.text, "html.parser")

# ── Step 1: Find the h3 with id="Retained_players" ────────────────────────────
retained_heading = soup.find("h3", id="Retained_players")

# ── Step 2: Grab only tables until the next h2 or h3 ─────────────────────────
retained_tables = []
for sibling in retained_heading.find_next_siblings():
    if sibling.name in ["h2", "h3"]:
        break
    if sibling.name == "table" and "wikitable" in sibling.get("class", []):
        retained_tables.append(sibling)
    for t in sibling.find_all("table", class_="wikitable"):
        if t not in retained_tables:
            retained_tables.append(t)

print(f"Found {len(retained_tables)} team tables")

# ── Step 3: Parse each team table ─────────────────────────────────────────────
all_rows = []

for table in retained_tables:
    caption = table.find("caption")
    if not caption:
        continue

    team_name = re.sub(r"\[\d+\]", "", caption.get_text(strip=True)).strip()

    header_row = table.find("tr")
    col_headers = [th.get_text(strip=True) for th in header_row.find_all("th")]

    if "Player" not in col_headers:
        continue

    for row in table.find_all("tr")[1:]:
        tds = row.find_all("td")
        if not tds:
            continue

        def cell_text(td):
            a = td.find("a")
            return a.get_text(strip=True) if a else td.get_text(strip=True)

        row_data = [cell_text(td) for td in tds]

        row_dict = {"Team": team_name}
        for i, col in enumerate(col_headers):
            row_dict[col] = row_data[i] if i < len(row_data) else None

        all_rows.append(row_dict)

# ── Step 4: Clean up ──────────────────────────────────────────────────────────
df = pd.DataFrame(all_rows)

keep_cols = [c for c in ["Team", "No.", "Player", "Nationality", "Salary"] if c in df.columns]
df = df[keep_cols]

df["Salary"] = df["Salary"].astype(str).str.extract(r"(₹[\d.]+ ?crore)")
df["Team"] = df["Team"].str.replace(r"\[\d+\]", "", regex=True).str.strip()
df["No."] = df["No."].astype(str).str.strip()

print(df.to_string(index=False))
df.to_csv("ipl_2025_retained_clean.csv", index=False)
print(f"\n✅ Saved {len(df)} players to ipl_2025_retained_clean.csv")

Found 0 team tables


KeyError: 'Salary'

In [25]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

URL = "https://en.wikipedia.org/wiki/List_of_2025_Indian_Premier_League_personnel_changes"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

resp = requests.get(URL, headers=HEADERS)
soup = BeautifulSoup(resp.text, "html.parser")

# ── Step 1: Find the heading div (parent of h3) ───────────────────────────────
retained_heading = soup.find("h3", id="Retained_players")
heading_div = retained_heading.parent  # the mw-heading div

# ── Step 2: Walk siblings of the heading div until next heading ───────────────
retained_tables = []
for sibling in heading_div.find_next_siblings():
    # stop at next heading (h2 or h3 level div)
    if sibling.name == "div" and "mw-heading" in sibling.get("class", []):
        break
    if sibling.name == "table" and "wikitable" in sibling.get("class", []):
        retained_tables.append(sibling)
    # tables may be nested inside divs
    for t in sibling.find_all("table", class_="wikitable"):
        if t not in retained_tables:
            retained_tables.append(t)

print(f"Found {len(retained_tables)} team tables")

# ── Step 3: Parse each team table ────────────────────────────────────────────
all_rows = []

for table in retained_tables:
    caption = table.find("caption")
    if not caption:
        continue

    team_name = re.sub(r"\[\d+\]", "", caption.get_text(strip=True)).strip()

    header_row = table.find("tr")
    col_headers = [th.get_text(strip=True) for th in header_row.find_all("th")]

    if "Player" not in col_headers:
        continue

    for row in table.find_all("tr")[1:]:
        tds = row.find_all("td")
        if not tds:
            continue

        def cell_text(td):
            a = td.find("a")
            return a.get_text(strip=True) if a else td.get_text(strip=True)

        row_data = [cell_text(td) for td in tds]

        row_dict = {"Team": team_name}
        for i, col in enumerate(col_headers):
            row_dict[col] = row_data[i] if i < len(row_data) else None

        all_rows.append(row_dict)

# ── Step 4: Clean up ──────────────────────────────────────────────────────────
df = pd.DataFrame(all_rows)
print("Columns found:", df.columns.tolist())
print(df.head())

# Rename whichever column has salary info
salary_col = [c for c in df.columns if "price" in c.lower() or "salary" in c.lower()]
if salary_col:
    df = df.rename(columns={salary_col[0]: "Salary"})

keep_cols = [c for c in ["Team", "No.", "Player", "Nationality", "Salary"] if c in df.columns]
df = df[keep_cols]

df["Salary"] = df["Salary"].astype(str).str.extract(r"(₹[\d.]+ ?crore)")
df["Team"] = df["Team"].str.replace(r"\[\d+\]", "", regex=True).str.strip()

print(df.to_string(index=False))
df.to_csv("ipl_2025_retained_clean.csv", index=False)
print(f"\n✅ Saved {len(df)} players to ipl_2025_retained_clean.csv")

Found 31 team tables
Columns found: ['Team', 'No.', 'Player', 'Nationality', 'Salary', 'Auctioned/retention price', 'Reason', 'Withdrawal announcement date', 'Replacement player', "Replacement player's price[74]", 'Signing date', 'Ref.']
                  Team No.     Player Nationality  Salary  \
0  Chennai Super Kings   1      India           ₹     NaN   
1  Chennai Super Kings   2      India           ₹     NaN   
2  Chennai Super Kings   3  Sri Lanka           ₹     NaN   
3  Chennai Super Kings   4      India           ₹     NaN   
4  Chennai Super Kings   5      India           ₹     NaN   

  Auctioned/retention price Reason Withdrawal announcement date  \
0                       NaN    NaN                          NaN   
1                       NaN    NaN                          NaN   
2                       NaN    NaN                          NaN   
3                       NaN    NaN                          NaN   
4                       NaN    NaN                          